# 2. Fit models and choose settings
This notebook contains the actual model creation, fitting loops and validation decisions.

We compare rules, Isolation Forest, LOF and One-Class SVM. At each stage, the best individual anomaly model on validation is combined with rules using a fixed 50/50 blend. The final comparison therefore has five approaches.

No test scores are calculated here. Candidate settings are selected by validation F1; tied settings keep the first candidate in the documented order.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "config/generator.yaml").exists(), "Open this notebook from the Research repository."
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
CONFIG = yaml.safe_load((ROOT / "config/generator.yaml").read_text())
OUT = ROOT / "report/core"
TABLES, FIGURES = OUT / "tables", OUT / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

In [2]:
from core_evaluation import load_data, split_stage, choose_threshold, band_cutoffs, fingerprint, HYBRID_WEIGHT
from models import STAGE_A_FEATURES, STAGE_B_FEATURES, isolation_forest, lof, one_class_svm, rule_score, hybrid_score
import joblib
from datetime import datetime, timezone
MODEL_SEED = 0
df = load_data(ROOT)
validation_records = []
split_counts = []
stage_bundles = {}
MODELS = OUT / "models"
MODELS.mkdir(exist_ok=True)

## Fit and select using validation only
For creation, periods follow creation time. For redemption, periods follow redemption time. Both use 90 fitting days, 30 validation days and 60 test days. A redemption that happens later cannot enter an earlier fitting period.

Anomaly fitting receives feature columns only. It does not receive the scam label. Validation labels are used to choose settings and alert thresholds, so the complete procedure is not label-free.

In [3]:
for stage in ("A", "B"):
    fit, validation, test = split_stage(df, stage, CONFIG["start_date"])
    for period, part in (("fit", fit), ("validation", validation), ("test", test)):
        split_counts.append({"stage": stage, "period": period, "vouchers": len(part)})
    features = STAGE_A_FEATURES if stage == "A" else STAGE_B_FEATURES
    y_validation = validation.is_scam.to_numpy()
    fitted, thresholds, settings, validation_scores = {}, {}, {}, {}

    # Rules do not need fitting. Points remain unchanged.
    rules = rule_score(validation, stage)[0]
    thresholds["Rules"], result = choose_threshold(y_validation, rules)
    validation_scores["Rules"] = rules
    validation_records.append({"stage": stage, "model": "Rules", "settings": "fixed points", **result})

    # Model creation is explicit here. Sample caps remain in the reusable model wrappers.
    candidates = {
        "Isolation Forest": [({"trees": 300, "samples_per_tree": 256}, isolation_forest(features, MODEL_SEED))],
        "LOF": [({"neighbors": k}, lof(features, k, MODEL_SEED)) for k in (20,50,100,200)],
        "One-Class SVM": [({"nu": nu}, one_class_svm(features, nu, MODEL_SEED)) for nu in (.01,.05,.1,.2)],
    }
    best_f1 = {}
    for name, choices in candidates.items():
        winner = None
        for parameters, scorer in choices:
            scorer.fit(fit[features])
            scores = scorer.score(validation[features])
            threshold, result = choose_threshold(y_validation, scores)
            validation_records.append({"stage": stage, "model": name, "settings": str(parameters), **result})
            if winner is None or result["f1"] > winner[0]:
                winner = (result["f1"], scorer, parameters, threshold, scores)
        best_f1[name], fitted[name], settings[name], thresholds[name], validation_scores[name] = winner

    # Dict order breaks exact ties: IF, then LOF, then SVM.
    selected_anomaly = max(best_f1, key=best_f1.get)
    hybrid = hybrid_score(rules, validation_scores[selected_anomaly], HYBRID_WEIGHT)
    thresholds["Hybrid"], result = choose_threshold(y_validation, hybrid)
    validation_records.append({"stage": stage, "model": "Hybrid", "settings": f"50% rules + 50% {selected_anomaly}", **result})
    stage_bundles[stage] = {"scorers": fitted, "settings": settings, "thresholds": thresholds,
        "hybrid_anomaly": selected_anomaly, "hybrid_weight": HYBRID_WEIGHT,
        "band_cutoffs": band_cutoffs(y_validation, hybrid, thresholds["Hybrid"])}
    print(f"Stage {stage}: hybrid uses {selected_anomaly}; alert threshold {thresholds['Hybrid']:.4f}")

Stage A: hybrid uses One-Class SVM; alert threshold 0.7167


Stage B: hybrid uses One-Class SVM; alert threshold 0.8157


## Save the choices before testing
The saved bundle contains the three fitted anomaly models and fixed thresholds for each stage. A fingerprint records the data, configuration and fitting code. Notebook 03 refuses to use a bundle if these have changed. UI risk bands reuse the existing recall/precision targets; they do not select the models.

In [4]:
selection = pd.DataFrame(validation_records)
selection.to_csv(TABLES / "validation_selection.csv", index=False)
pd.DataFrame(split_counts).to_csv(TABLES / "split_counts.csv", index=False)
bundle = {"fingerprint": fingerprint(ROOT), "created_at": datetime.now(timezone.utc).isoformat(),
    "model_seed": MODEL_SEED, "simulator_seed": CONFIG["seed"], "stages": stage_bundles,
    "protocol": "Fit days 0–89; validate 90–119; test 120–179; periods use the scoring time."}
joblib.dump(bundle, MODELS / "fitted_models.joblib")
display(selection[["stage", "model", "settings", "precision", "recall", "f1", "threshold"]].round(4))
print("Fitting and validation complete. Run notebook 03 to evaluate the fixed choices.")

,stage,model,settings,precision,recall,f1,threshold
0,A,Rules,fixed points,0.4159,0.3240,0.3642,60.0000
1,A,Isolation Forest,"{'trees': 300, 'samples_per_tree': 256}",0.4520,0.3352,0.3849,0.9894
2,A,LOF,{'neighbors': 20},0.2816,0.2378,0.2579,0.9910
3,A,LOF,{'neighbors': 50},0.3808,0.2154,0.2751,0.9937
4,A,LOF,{'neighbors': 100},0.3413,0.2678,0.3001,0.9904
5,A,LOF,{'neighbors': 200},0.2674,0.2884,0.2775,0.9864
6,A,One-Class SVM,{'nu': 0.01},0.3171,0.3408,0.3285,0.9882
7,A,One-Class SVM,{'nu': 0.05},0.4028,0.3801,0.3911,0.9884
8,A,One-Class SVM,{'nu': 0.1},0.4092,0.3839,0.3961,0.9882
9,A,One-Class SVM,{'nu': 0.2},0.3069,0.4494,0.3647,0.9794


Fitting and validation complete. Run notebook 03 to evaluate the fixed choices.
